# R07 Final Model — Performance & SHAP Analysis

**Model**: R07 (`S2_Reduced`) — LightGBM multiclass signal detector  
**Target**: 5-day excess return of TD vs XFN sector ETF (`target_excess_xfn_5d`)  
**Labels**: Outperform (+1) / Neutral (0) / Underperform (−1), band ±0.30%  
**Features**: 30 curated event-style + market/macro features  
**Validation**: 7-fold expanding-window walk-forward (stride-5 offset averaging)

This notebook:
1. Loads the locked R07 artifact and verifies it
2. Re-runs the 7-fold walk-forward evaluation
3. Reports per-fold and aggregate performance (DirAcc, active sign accuracy, coverage)
4. Generates SHAP analysis (global importance + fold heatmap)

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import shap

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path('').resolve()
ROOT = NOTEBOOK_DIR.parents[1]          # project root
EXP_PARENT = ROOT / 'step3_predictive_model/model_experiments'

for p in [str(ROOT), str(EXP_PARENT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

ARTIFACT_DIR = NOTEBOOK_DIR / 'artifacts'

print(f'ROOT: {ROOT}')
print(f'Artifact dir: {ARTIFACT_DIR}')

## 1. Load & Verify Artifact

In [ ]:
import pickle, json

pkl_files = sorted(ARTIFACT_DIR.glob('r07-current-best-*.pkl'))
assert pkl_files, 'No R07 artifact found in artifacts/'
pkl_path = pkl_files[-1]   # most recent
json_path = pkl_path.with_suffix('.json')

with open(pkl_path, 'rb') as f:
    artifact = pickle.load(f)

with open(json_path) as f:
    meta = json.load(f)

model     = artifact['model']
features  = artifact['features']
TARGET    = artifact['target']
THRESHOLD = artifact['threshold']
label_map = artifact['label_map']       # {-1:0, 0:1, 1:2}
inv_map   = artifact['inverse_label_map']

assert meta['features'] == features, 'Feature mismatch between pkl and json!'

print(f'Artifact : {pkl_path.name}')
print(f'Target   : {TARGET}')
print(f'Threshold: ±{THRESHOLD*100:.1f}%')
print(f'Features : {len(features)}')
print(f'Model    : {model}')

## 2. Load Dataset

In [ ]:
from redesign_single_stock.src.run_redesign_experiments import load_dataset, label_3class
from src.models.walk_forward_config import E10_FOLDS, STRIDE_EVAL

df = load_dataset()
print(f'Dataset: {len(df)} rows, {df["date"].min().date()} – {df["date"].max().date()}')
print(f'Missing features: {[f for f in features if f not in df.columns]}')

## 3. Walk-Forward Evaluation

In [ ]:
import lightgbm as lgb
from sklearn.metrics import f1_score

STRIDE  = STRIDE_EVAL
OFFSETS = list(range(STRIDE))


def fit_predict(train_df, test_df):
    x_tr = train_df[features].fillna(train_df[features].median())
    y_tr = label_3class(train_df[TARGET].values, threshold=THRESHOLD)
    y_enc = np.array([label_map[v] for v in y_tr], dtype=int)

    clf = lgb.LGBMClassifier(
        objective='multiclass', num_class=3,
        n_estimators=120, learning_rate=0.05, num_leaves=8,
        min_child_samples=40, feature_fraction=0.8,
        reg_alpha=0.2, reg_lambda=1.0,
        random_state=42, verbose=-1, n_jobs=1,
    )
    clf.fit(x_tr, y_enc)

    x_te  = test_df[features].fillna(train_df[features].median())
    y_te  = label_3class(test_df[TARGET].values, threshold=THRESHOLD)
    preds_enc = clf.predict(x_te)
    preds = np.array([inv_map[p] for p in preds_enc])
    return clf, preds, y_te


def offset_metrics(preds, y_true, dates):
    rows = []
    for off in OFFSETS:
        idx = np.arange(off, len(dates), STRIDE)
        p, y = preds[idx], y_true[idx]
        active = (p != 0)
        valid_active = active.sum() > 0
        rows.append({
            'mean_acc': (p == y).mean(),
            'macro_f1': f1_score(y, p, average='macro', zero_division=0, labels=[-1, 0, 1]),
            'active_cov': active.mean(),
            'active_asa': np.nanmean(np.sign(p[active]) == np.sign(y[active])) if valid_active else np.nan,
        })
    r = pd.DataFrame(rows).mean()
    return r


fold_results = []
fold_models  = []
fold_shap_dfs = []

for fold_id, train_end, test_start, test_end in E10_FOLDS:
    train_mask = (df['date'] <= pd.Timestamp(train_end)) & df[TARGET].notna()
    test_mask  = (df['date'] >= pd.Timestamp(test_start)) & (df['date'] <= pd.Timestamp(test_end)) & df[TARGET].notna()
    train_df, test_df = df[train_mask], df[test_mask]

    clf, preds, y_true = fit_predict(train_df, test_df)
    m = offset_metrics(preds, y_true, test_df['date'].values)

    # SHAP on test set
    explainer  = shap.TreeExplainer(clf)
    x_te_clean = test_df[features].fillna(train_df[features].median())
    shap_vals  = explainer.shap_values(x_te_clean)   # list of 3 arrays
    # mean |SHAP| across classes and samples
    mean_abs   = np.mean([np.abs(sv).mean(axis=0) for sv in shap_vals], axis=0)
    shap_ser   = pd.Series(mean_abs, index=features, name=fold_id)
    fold_shap_dfs.append(shap_ser)

    fold_results.append({
        'fold': fold_id, 'train_rows': len(train_df), 'test_rows': len(test_df),
        'train_end': train_end, 'test_start': test_start, 'test_end': test_end,
        **m.to_dict()
    })
    fold_models.append(clf)
    print(f'Fold {fold_id}  mean_acc={m.mean_acc:.3f}  asa={m.active_asa:.3f}  cov={m.active_cov:.3f}')

results_df = pd.DataFrame(fold_results).set_index('fold')
print('\nDone.')

## 4. Performance Summary

In [ ]:
display_cols = ['train_rows', 'test_rows', 'mean_acc', 'macro_f1', 'active_cov', 'active_asa']

print('=== Per-Fold Performance ===')
display(results_df[display_cols].round(3))

means = results_df[display_cols].mean()
print('\n=== 7-Fold Mean ===')
print(f'  Mean accuracy        : {means.mean_acc:.3f}')
print(f'  Macro F1             : {means.macro_f1:.3f}')
print(f'  Active coverage      : {means.active_cov:.3f}')
print(f'  Active sign accuracy : {means.active_asa:.3f}')

# DirAcc metrics
results_df['DirAcc_strict']   = results_df['active_asa'] * results_df['active_cov']
results_df['DirAcc_abstain50'] = (results_df['active_asa'] * results_df['active_cov']
                                  + 0.5 * (1 - results_df['active_cov']))

print('\n=== DirAcc Metrics (7-fold means) ===')
print(f'  DirAcc_strict   (active-weighted) : {results_df["DirAcc_strict"].mean():.3f}')
print(f'  DirAcc_abstain50 (abstain=50/50)  : {results_df["DirAcc_abstain50"].mean():.3f}')
print(f'  Naive max-bias baseline           : ~0.520  (long-share on excess-return target)')

## 5. SHAP Global Feature Importance

In [ ]:
shap_df   = pd.DataFrame(fold_shap_dfs)           # (7 folds, 30 features)
global_imp = shap_df.mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(global_imp.index[::-1], global_imp.values[::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP| across folds and classes')
ax.set_title('R07 — Global Feature Importance (SHAP)', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
plt.tight_layout()
plt.show()

print('\nTop-10 features by mean |SHAP|:')
print(global_imp.head(10).to_string())

## 6. SHAP Stability Across Folds (Heatmap)

In [ ]:
# Rank features by global mean, show top 20
top20 = global_imp.head(20).index.tolist()
heat  = shap_df[top20].T   # (features, folds)

fig, ax = plt.subplots(figsize=(11, 7))
im = ax.imshow(heat.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels([f'v{i+1}' for i in range(len(heat.columns))], fontsize=10)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(top20, fontsize=9)
ax.set_xlabel('Fold')
ax.set_title('R07 — SHAP Importance per Fold (Top-20 features)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='Mean |SHAP|')
plt.tight_layout()
plt.show()

## 7. SHAP Beeswarm on Final Model (trained on all data)

In [ ]:
# Use the locked final model (trained on full dataset) for beeswarm
df_clean = df.dropna(subset=[TARGET])
x_all = df_clean[features].fillna(df_clean[features].median())

explainer_final = shap.TreeExplainer(model)
shap_vals_final = explainer_final.shap_values(x_all)

# Outperform class (index 2 -> label +1)
shap.summary_plot(
    shap_vals_final[2], x_all,
    feature_names=features,
    plot_type='dot',
    max_display=20,
    show=False,
)
plt.title('R07 — SHAP Beeswarm (Outperform class, full dataset)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Feature List (Audit)

All 30 R07 features grouped by block for reproducibility.

In [ ]:
feature_groups = {
    'Market / Price': ['td_return_5d', 'td_return_20d', 'td_volatility_20d',
                       'td_vs_xfn_5d', 'td_vs_tsx_5d', 'td_corr_pv_20d',
                       'td_volume_change_20d', 'td_dist_52w_high'],
    'Rates / Macro':  ['yield_curve_slope', 'yield_10y_level'],
    'Global Risk':    ['vix_volatility_20d', 'dxy_level', 'gold_level', 'fx_usdcad_level'],
    'News Flow':      ['news_sent_mean_30d', 'news_count_30d', 'days_since_last_news'],
    'Timing':         ['days_since_call', 'is_earnings_week'],
    'NLP Event':      ['evt_ceo_tone', 'evt_cfo_tone', 'evt_framing_gap',
                       'evt_aml_pressure', 'evt_aml_shift', 'evt_guidance_strength',
                       'evt_guidance_shift', 'evt_macro_topic', 'evt_topic_entropy',
                       'evt_news_tone', 'evt_news_flow'],
}

all_listed = sum(feature_groups.values(), [])
assert sorted(all_listed) == sorted(features), 'Feature group mismatch!'

for group, feats in feature_groups.items():
    print(f'\n{group} ({len(feats)} features):')
    for f in feats:
        print(f'  - {f}')

print(f'\nTotal: {len(features)} features ✓')